# Jani — entrenamiento del clasificador de hojas de café

**Cómo usarlo (no hay que editar nada):**
1. En Colab: `Archivo → Subir notebook` y elige este archivo.
2. `Entorno de ejecución → Cambiar tipo de entorno → GPU T4 → Guardar`.
3. `Entorno de ejecución → Ejecutar todo`. Tarda unos **40–70 minutos** (entrena dos modelos y elige el mejor).
4. Al final el navegador descarga **`jani_model.zip`**. Descomprímelo **dentro de** `app/public/models/arabica-v1/`
   del repositorio, reemplazando lo que haya, y en `app/` ejecuta `npm run check:assets` y `npm run build`.

**Qué hace:** descarga los datos, entrena `efficientnet_lite0` y `mobilenetv3_small_100`, calibra la confianza, elige el
umbral de "no estoy segura" con datos de otros países, exporta a ONNX, cuantiza a int8, compara y se queda con el mejor
modelo que pese menos de 10 MB.

**Qué trae el zip:** un solo archivo `.onnx` (el recomendado), `model_card.json` (clases, normalización, temperatura,
umbrales, métricas y fuentes) y `confusion_*.png` para el video.

Si una descarga de Mendeley falla, el mensaje dice qué página abrir y con qué nombre subir el `.zip` a Colab (ícono de
carpeta, a la izquierda); luego `Ejecutar todo` otra vez. Si Uganda/Perú no se pueden bajar, el modelo igual se entrena,
pero sin el número honesto de precisión fuera de distribución.

In [ ]:
!pip -q install timm datasets onnx onnxruntime onnxscript scikit-learn
!apt-get -qq install -y unrar > /dev/null 2>&1 || true

## Configuración
No hace falta cambiar nada. Los valores están pensados para una GPU T4 de Colab gratis.

In [ ]:
# Modelos que se entrenan en esta ejecución. Al final se elige el mejor automáticamente.
MODELS = ["efficientnet_lite0", "mobilenetv3_small_100"]

IMG_SIZE = 224
EPOCHS = 8
BATCH = 64
LR = 1e-3
# JMuBEN está desbalanceado (Sana 18.983, Minador 16.978 ... Phoma 6.571). Con este tope se usan TODAS las fotos de
# la clase más chica y la misma cantidad de las demás (balanceado). None = todas las fotos (desbalanceado, más lento).
MAX_PER_CLASS = 6500
USE_BRACOL = True        # suma hojas de Brasil (hojas completas + recortes de síntomas)
SEED = 42

# Orden canónico de clases. La app depende de este orden.
CLASSES = ["sana", "roya", "minador", "phoma", "cercospora"]

# Regla de "no estoy segura". El umbral de confianza se elige solo con datos externos (sección 6):
# el más bajo que logre TARGET_PRECISION respondiendo al menos MIN_COVERAGE de las veces.
TARGET_PRECISION = 0.90
MIN_COVERAGE = 0.30
DEFAULT_CONF = 0.70      # se usa si no hay datos externos
MARGIN_THRESHOLD = 0.15

MAX_MODEL_MB = 10        # criterio del reto
MAX_INT8_DROP = 0.02     # caída de precisión aceptable al cuantizar

OUT_DIR = "jani_model"
SMOKE_TEST = False       # True solo para probar el notebook sin GPU con imágenes sintéticas

In [ ]:
import os, re, io, csv, json, glob, math, random, zipfile, shutil, subprocess, urllib.request, time
from collections import Counter
import numpy as np, torch, torch.nn as nn, timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib
import matplotlib.pyplot as plt

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)
if SMOKE_TEST:
    EPOCHS, BATCH, MAX_PER_CLASS = 1, 8, None
    matplotlib.use("Agg"); plt.show = lambda *a, **k: None
else:
    assert device == "cuda", "Activa la GPU: Entorno de ejecución → Cambiar tipo de entorno → T4"
NUM_WORKERS = 0 if SMOKE_TEST else 2

## 1. Datos de entrenamiento: JMuBEN (Kenia)

Espejo en Hugging Face (`Project-AgML/arabica_coffee_leaf_disease_classification`, CC BY 4.0, 58.549 imágenes, ~1,8 GB).

In [ ]:
def to_canonical(name):
    """Nombre de carpeta o etiqueta -> clase canónica, o None si no es una de las 5."""
    n = re.sub(r"[^a-z]+", " ", str(name).lower()).strip()
    w = set(n.split())
    if w & {"unhealthy", "diseased", "enferma", "enfermas"}: return None
    if re.search(r"cer?s?cospora|cescospora", n) or ("brown" in w and "eye" in w): return "cercospora"
    if "phoma" in w or ("brown" in w and ("leaf" in w or "spot" in w)): return "phoma"   # BRACOL: "brown leaf spot"
    if w & {"rust", "roya", "clr"}: return "roya"
    if w & {"miner", "minador"}: return "minador"
    if w & {"healthy", "health", "sana", "sano", "sanas", "saludable"}: return "sana"
    return None

if SMOKE_TEST:
    # Imágenes sintéticas: cada clase con un color dominante distinto (solo para probar el código).
    hf_names = ["Cerscospora", "Healthy", "Leaf_rust", "Miner", "Phoma"]
    _cols = [(150, 90, 40), (40, 160, 60), (220, 140, 30), (200, 200, 170), (90, 60, 40)]
    _rng = np.random.default_rng(0)
    def _fake(c):
        a = np.clip(np.array(_cols[c]) + _rng.normal(0, 25, (96, 96, 3)), 0, 255).astype("uint8")
        return Image.fromarray(a)
    _fake_rows = [{"image": _fake(c), "label": c} for c in range(5) for _ in range(16)]
    labels_all = np.array([r["label"] for r in _fake_rows])
    def hf_row(i): return _fake_rows[i]
else:
    from datasets import load_dataset
    ds = load_dataset("Project-AgML/arabica_coffee_leaf_disease_classification", split="train")
    hf_names = ds.features["label"].names
    labels_all = np.array(ds["label"])
    def hf_row(i): return ds[int(i)]

print("Clases en el dataset:", hf_names, "| total:", len(labels_all))
unmapped = [n for n in hf_names if to_canonical(n) is None]
assert not unmapped, f"Clases sin mapear: {unmapped}. Ajusta to_canonical()."
hf_to_idx = {i: CLASSES.index(to_canonical(n)) for i, n in enumerate(hf_names)}
print("Mapeo:", {hf_names[i]: CLASSES[j] for i, j in hf_to_idx.items()})

train_idx, val_idx = [], []
rng = np.random.default_rng(SEED)
for c in range(len(hf_names)):
    idx = rng.permutation(np.where(labels_all == c)[0])
    if MAX_PER_CLASS: idx = idx[:MAX_PER_CLASS]
    n_val = max(1, int(0.15 * len(idx)))
    val_idx += idx[:n_val].tolist(); train_idx += idx[n_val:].tolist()
print("Entrenamiento:", len(train_idx), "| Validación:", len(val_idx))

## 2. Datos externos (Mendeley Data)

- **BRACOL** (Brasil, CC BY 4.0): se suma al entrenamiento. Trae hojas completas (etiquetas en `dataset.csv`, columna
  `predominant_stress`: 0 sana, 1 minador, 2 roya, 3 phoma, 4 cercospora) y recortes de síntomas (carpetas `1_health`…).
- **Uganda** y **Perú**: solo para la **prueba fuera de distribución**. El modelo nunca los ve al entrenar.

Si una descarga falla, abre la página del dataset, pulsa *Download All* y sube el `.zip` a Colab con el nombre que indica
el mensaje.

In [ ]:
MENDELEY = {
    "bracol": ("yy2k5y8mxg", 1, "https://data.mendeley.com/datasets/yy2k5y8mxg/1"),
    "uganda": ("k36wnd6knb", 1, "https://data.mendeley.com/datasets/k36wnd6knb/1"),
    "peru":   ("mfpxg4y65r", 1, "https://data.mendeley.com/datasets/mfpxg4y65r/1"),
}
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
STRESS_CODES = {0: "sana", 1: "minador", 2: "roya", 3: "phoma", 4: "cercospora"}   # BRACOL predominant_stress

def extract_nested(folder):
    for _ in range(3):
        found = False
        for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True):
            low = p.lower()
            try:
                if low.endswith(".zip"):
                    zipfile.ZipFile(p).extractall(os.path.splitext(p)[0]); os.remove(p); found = True
                elif low.endswith(".rar"):
                    d = os.path.splitext(p)[0]; os.makedirs(d, exist_ok=True)
                    subprocess.run(["unrar", "x", "-o+", "-inul", p, d + "/"]); os.remove(p); found = True
            except Exception as e:
                print("  No se pudo extraer", p, e)
        if not found: break

def fetch_mendeley(key):
    did, ver, page = MENDELEY[key]
    dest = f"data/{key}"
    if os.path.isdir(dest) and any(f.lower().endswith(IMG_EXT) for _, _, fs in os.walk(dest) for f in fs):
        return dest
    os.makedirs(dest, exist_ok=True)
    local_zip = f"{key}.zip"
    if not os.path.exists(local_zip):
        url = f"https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/{did}-{ver}.zip"
        try:
            print(f"Descargando {key} ...")
            urllib.request.urlretrieve(url, local_zip)
        except Exception as e:
            print(f"  Descarga automática falló ({e}).")
            print(f"  Abre {page} → Download All → sube el archivo a Colab como '{local_zip}' y vuelve a ejecutar.")
            return None
    try:
        zipfile.ZipFile(local_zip).extractall(dest)
    except Exception as e:
        print("  Zip inválido:", e); return None
    extract_nested(dest)
    return dest

def scan_folder(root):
    """(ruta, clase canónica o 'desconocida') usando los nombres de las carpetas."""
    items = []
    for dp, _, fs in os.walk(root):
        parts = os.path.relpath(dp, root).split(os.sep)
        label = next((l for l in (to_canonical(p) for p in reversed(parts)) if l), None)
        for f in fs:
            if f.lower().endswith(IMG_EXT):
                items.append((os.path.join(dp, f), label or "desconocida"))
    return items

def csv_labels(root):
    """Etiquetas de BRACOL (hojas completas): dataset*.csv con columnas id y predominant_stress."""
    by_path = {}
    for path in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
        try:
            rows = list(csv.DictReader(open(path, encoding="utf-8-sig")))
        except Exception:
            continue
        if not rows or not {"id", "predominant_stress"} <= set(rows[0].keys()): continue
        folder = os.path.dirname(path)
        img_dirs = [d for d in glob.glob(os.path.join(folder, "**"), recursive=True) if os.path.isdir(d)]
        index = {}
        for d in img_dirs:
            for f in os.listdir(d):
                stem, ext = os.path.splitext(f)
                if ext.lower() in IMG_EXT and os.path.basename(d).lower() in ("leaf", "leaves", "images", "leaf_images"):
                    index[stem] = os.path.join(d, f)
        for r in rows:
            p = index.get(str(r["id"]).strip())
            code = str(r["predominant_stress"]).strip()
            if p and code.isdigit() and int(code) in STRESS_CODES:
                by_path[p] = STRESS_CODES[int(code)]
    return by_path

def summarize(name, items):
    print(f"{name}: {len(items)} imágenes →", dict(Counter(l for _, l in items)))

def make_smoke_external():
    """Copias sintéticas con la misma estructura que los datasets reales (solo SMOKE_TEST)."""
    out = {}
    def save(path, c):
        os.makedirs(os.path.dirname(path), exist_ok=True); _fake(c).save(path)
    names = ["Cerscospora", "Healthy", "Leaf_rust", "Miner", "Phoma"]   # índice = color de _fake
    # BRACOL: hojas completas con CSV + recortes en carpetas 1_health…5_cercospora
    root = "data/smoke_bracol"; shutil.rmtree(root, ignore_errors=True)
    code_to_fake = {0: 1, 1: 3, 2: 2, 3: 4, 4: 0}
    os.makedirs(f"{root}/dataset/leaf", exist_ok=True)
    with open(f"{root}/dataset/dataset.csv", "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(["id", "predominant_stress", "miner", "rust", "phoma", "cercospora", "severity"])
        for i in range(10):
            code = i % 5; save(f"{root}/dataset/leaf/{i+1}.jpg", code_to_fake[code]); w.writerow([i + 1, code, 0, 0, 0, 0, 0])
    for k, folder in enumerate(["1_health", "2_miner", "3_rust", "4_phoma", "5_cercospora"]):
        for i in range(3): save(f"{root}/dataset/symptom/train/{folder}/{i}.jpg", code_to_fake[k])
    out["bracol"] = root
    for key, folders in [("uganda", ["healthy", "rust", "phoma"]), ("peru", ["sana", "roya", "ojo de gallo"])]:
        root = f"data/smoke_{key}"; shutil.rmtree(root, ignore_errors=True)
        for folder in folders:
            lab = to_canonical(folder)
            c = names.index({"sana": "Healthy", "roya": "Leaf_rust", "phoma": "Phoma"}[lab]) if lab else 3
            for i in range(8): save(f"{root}/{folder}/{i}.jpg", c)
        out[key] = root
    return out

smoke_dirs = make_smoke_external() if SMOKE_TEST else {}
ext = {}
for key in MENDELEY:
    if key == "bracol" and not USE_BRACOL: continue
    d = smoke_dirs.get(key) if SMOKE_TEST else fetch_mendeley(key)
    items = scan_folder(d) if d else []
    if d and key == "bracol":
        labels = csv_labels(d)
        items = [(p, labels.get(p, l)) for p, l in items]
        print(f"  BRACOL: {len(labels)} hojas completas etiquetadas desde el CSV")
    ext[key] = items
    summarize(key, items)
print("\nRevisa los conteos. Si una clase conocida aparece como 'desconocida', ajusta to_canonical().")

## 3. Datasets y aumentación

La app estira la foto completa a 224×224 sin recortar y normaliza con la media y desviación del modelo. La evaluación hace
exactamente lo mismo; el entrenamiento agrega recortes, giros, cambios de color, perspectiva y desenfoque para parecerse
a fotos de campo.

In [ ]:
def make_transforms(mean, std):
    train_tf = T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), ratio=(0.6, 1.67)),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(25),
        T.RandomPerspective(0.2, p=0.3),
        T.ColorJitter(0.35, 0.35, 0.3, 0.05),
        T.RandomApply([T.GaussianBlur(5)], p=0.2),
        T.ToTensor(), T.Normalize(mean, std),
    ])
    eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(mean, std)])
    return train_tf, eval_tf

class HFSet(Dataset):
    def __init__(self, idx, tf): self.idx, self.tf = idx, tf
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        r = hf_row(self.idx[i])
        return self.tf(r["image"].convert("RGB")), hf_to_idx[r["label"]]

class FileSet(Dataset):
    def __init__(self, items, tf): self.items, self.tf = items, tf
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        for k in range(10):   # si una imagen está dañada, usa otra
            p, l = self.items[(i + k) % len(self.items)]
            try:
                return self.tf(Image.open(p).convert("RGB")), CLASSES.index(l)
            except Exception:
                continue
        raise RuntimeError(f"No se pudo leer ninguna imagen cerca de {self.items[i][0]}")

bracol_known = [(p, l) for p, l in ext.get("bracol", []) if l in CLASSES]
print("BRACOL para entrenar:", len(bracol_known), dict(Counter(l for _, l in bracol_known)))

# Datos externos: mitad para elegir el umbral (calib) y mitad para reportar (holdout), por clase.
def split_half(items, seed=SEED):
    r = random.Random(seed); calib, hold = [], []
    by = {}
    for it in items: by.setdefault(it[1], []).append(it)
    for lab in sorted(by):
        group = by[lab][:]; r.shuffle(group); h = len(group) // 2
        calib += group[:h]; hold += group[h:]
    return calib, hold

ood = {}
for key in ["uganda", "peru"]:
    items = ext.get(key, [])
    known = [(p, l) for p, l in items if l in CLASSES]
    unknown = [(p, "sana") for p, l in items if l not in CLASSES]   # etiqueta ficticia: solo se mide la abstención
    if known:
        calib, hold = split_half(known)
        ood[key] = {"calib": calib, "hold": hold, "unknown": unknown}
        print(f"{key}: {len(calib)} para elegir umbral, {len(hold)} para reportar, {len(unknown)} de clases que no conoce")
if not ood:
    print("SIN datos externos: el umbral queda en DEFAULT_CONF y no habrá número honesto (revisa la sección 2).")

## 4. Entrenar, calibrar, evaluar y exportar cada modelo

Para cada modelo: entrenamiento → calibración de temperatura → umbral de duda con datos externos → métricas fuera de
distribución → ONNX fp32 → int8 → elección del archivo (≤ 10 MB).

In [ ]:
CONF_GRID = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

def get_logits(m, dl):
    m.eval(); L, Y = [], []
    with torch.no_grad():
        for x, y in dl:
            L.append(m(x.to(device)).float().cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)

def rule(logits, T, conf, margin=MARGIN_THRESHOLD):
    p = torch.softmax(logits / T, 1)
    top2 = p.topk(2, 1).values
    sure = (top2[:, 0] >= conf) & ((top2[:, 0] - top2[:, 1]) >= margin)
    return p.argmax(1), sure

def report(name, logits, y, T, conf):
    pred, sure = rule(logits, T, conf)
    acc_all = (pred == y).float().mean().item()
    cov = sure.float().mean().item()
    acc_sure = (pred[sure] == y[sure]).float().mean().item() if sure.any() else None
    txt = f"{acc_sure:.3f}" if acc_sure is not None else "—"
    print(f"  {name}: precisión total {acc_all:.3f} | responde en {cov:.1%} | precisión cuando responde {txt} | n={len(y)}")
    return {"n": int(len(y)), "accuracy_all": acc_all, "coverage": cov, "accuracy_when_answering": acc_sure, "escalation_rate": 1 - cov}

def choose_conf(logits, y, T):
    rows = []
    for c in CONF_GRID:
        pred, sure = rule(logits, T, c)
        cov = sure.float().mean().item()
        prec = (pred[sure] == y[sure]).float().mean().item() if sure.any() else 0.0
        rows.append((c, cov, prec))
    print("  umbral | responde | precisión cuando responde")
    for c, cov, prec in rows: print(f"   {c:.2f}  |  {cov:6.1%}  |  {prec:.3f}")
    ok = [r for r in rows if r[1] >= MIN_COVERAGE and r[2] >= TARGET_PRECISION]
    if ok: return ok[0][0]
    covered = [r for r in rows if r[1] >= MIN_COVERAGE]
    return max(covered, key=lambda r: r[2])[0] if covered else DEFAULT_CONF

def plot_cm(out, name, y, pred):
    cm = confusion_matrix(y, pred, labels=list(range(len(CLASSES))))
    fig, ax = plt.subplots(figsize=(5, 4)); ax.imshow(cm, cmap="Greens")
    ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)): ax.text(j, i, cm[i, j], ha="center", va="center")
    ax.set_xlabel("Predicción"); ax.set_ylabel("Real"); ax.set_title(name)
    plt.tight_layout(); plt.savefig(f"{out}/confusion_{name}.png", dpi=120); plt.show(); plt.close(fig)

def ort_session(path, optimized=False):
    # Se evalúa SIN optimizaciones de grafo: así da los mismos números que onnxruntime-web (WASM) en el teléfono.
    # Con optimizaciones, el onnxruntime de PC fusiona el int8 en kernels propios y puede dar otros resultados
    # (medido: hasta 5 logits de diferencia con int8 por canal; sin optimizar coincide con la app en < 0,5).
    import onnxruntime as ort
    so = ort.SessionOptions()
    if not optimized: so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    return ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])

def onnx_logits(path, dataset, n=1500):
    sess = ort_session(path)
    idx = random.Random(SEED).sample(range(len(dataset)), min(n, len(dataset)))
    L, Y, t = [], [], 0.0
    for i in idx:
        x, y = dataset[i]; t0 = time.time()
        L.append(sess.run(None, {"input": x.unsqueeze(0).numpy()})[0][0]); t += time.time() - t0; Y.append(y)
    return torch.tensor(np.array(L)), torch.tensor(Y), 1000 * t / max(1, len(idx))

def run_model(name):
    print(f"\n==================== {name} ====================")
    out = f"{OUT_DIR}/{name}"; os.makedirs(out, exist_ok=True)
    model = timm.create_model(name, pretrained=True, num_classes=len(CLASSES)).to(device)
    cfg = timm.data.resolve_data_config({}, model=model)
    mean, std = [float(v) for v in cfg["mean"]], [float(v) for v in cfg["std"]]
    print("Normalización:", mean, std)
    train_tf, eval_tf = make_transforms(mean, std)

    sets = [HFSet(train_idx, train_tf)] + ([FileSet(bracol_known, train_tf)] if bracol_known else [])
    train_dl = DataLoader(torch.utils.data.ConcatDataset(sets), BATCH, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    val_ds = HFSet(val_idx, eval_tf)
    val_dl = DataLoader(val_ds, BATCH, num_workers=NUM_WORKERS)

    # --- Entrenamiento ---
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl))
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best = -1.0
    for ep in range(EPOCHS):
        model.train(); t0 = time.time(); tot = 0.0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device, enabled=use_amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item()
        lg, yy = get_logits(model, val_dl)
        acc = (lg.argmax(1) == yy).float().mean().item()
        print(f"Época {ep+1}/{EPOCHS} | pérdida {tot/len(train_dl):.3f} | validación {acc:.4f} | {time.time()-t0:.0f}s")
        if acc >= best:
            best = acc; torch.save(model.state_dict(), f"{out}/best.pt")
    model.load_state_dict(torch.load(f"{out}/best.pt", map_location=device)); model.eval()
    print("Mejor validación (inflada por la aumentación de JMuBEN):", round(best, 4))

    # --- Calibración de temperatura (con validación) ---
    val_logits, val_y = get_logits(model, val_dl)
    logT = torch.zeros(1, requires_grad=True)
    o = torch.optim.LBFGS([logT], lr=0.1, max_iter=100)
    def _closure():
        o.zero_grad(); l = nn.functional.cross_entropy(val_logits / logT.exp(), val_y); l.backward(); return l
    o.step(_closure)
    temperature = float(logT.exp())
    if not math.isfinite(temperature) or temperature <= 0: temperature = 1.0
    print("Temperatura:", round(temperature, 3))

    # --- Umbral de duda con la mitad 'calib' de los datos externos ---
    ood_logits = {}
    for key, parts in ood.items():
        ood_logits[key] = {k: get_logits(model, DataLoader(FileSet(parts[k], eval_tf), BATCH, num_workers=NUM_WORKERS))
                           for k in ("calib", "hold") if parts[k]}
    calib = [v["calib"] for v in ood_logits.values() if "calib" in v]
    if calib:
        print("Elección del umbral (datos externos, mitad 'calib'):")
        conf = choose_conf(torch.cat([c[0] for c in calib]), torch.cat([c[1] for c in calib]), temperature)
    else:
        conf = DEFAULT_CONF
    print("Umbral de confianza elegido:", conf)

    # --- Métricas ---
    metrics = {"in_distribution_val": report("Validación (dentro de distribución)", val_logits, val_y, temperature, conf)}
    plot_cm(out, "validacion", val_y.numpy(), val_logits.argmax(1).numpy())
    for key, parts in ood_logits.items():
        if "hold" in parts:
            lg, yy = parts["hold"]
            metrics[f"ood_{key}_holdout"] = report(f"{key} fuera de distribución (mitad no usada para el umbral)", lg, yy, temperature, conf)
            plot_cm(out, key, yy.numpy(), lg.argmax(1).numpy())
            present = sorted(set(yy.tolist()))
            print(classification_report(yy, lg.argmax(1), labels=present, target_names=[CLASSES[i] for i in present], zero_division=0))
        unknown = ood[key]["unknown"]
        if unknown:
            lg, _ = get_logits(model, DataLoader(FileSet(unknown, eval_tf), BATCH, num_workers=NUM_WORKERS))
            _, sure = rule(lg, temperature, conf)
            rate = 1 - sure.float().mean().item()
            metrics[f"ood_{key}_unknown_class_abstention"] = {"n": len(unknown), "abstention_rate": rate}
            print(f"  {key}: ante {len(unknown)} fotos de clases que no conoce, dice 'no estoy segura' en {rate:.1%} (ideal ~100%)")

    # --- ONNX fp32 + int8 ---
    import onnx
    from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat
    model_cpu = model.to("cpu").eval()
    fp32, int8 = f"{out}/model_fp32.onnx", f"{out}/model_int8.onnx"
    kw = dict(input_names=["input"], output_names=["logits"], opset_version=17)
    try:
        torch.onnx.export(model_cpu, torch.randn(1, 3, IMG_SIZE, IMG_SIZE), fp32, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(model_cpu, torch.randn(1, 3, IMG_SIZE, IMG_SIZE), fp32, **kw)
    onnx.checker.check_model(onnx.load(fp32))

    class Calib(CalibrationDataReader):
        def __init__(self, n=300):
            sub = random.Random(SEED).sample(train_idx, min(n, len(train_idx)))
            self.it = iter([{"input": HFSet([i], eval_tf)[0][0].unsqueeze(0).numpy()} for i in sub])
        def get_next(self): return next(self.it, None)

    int8_ok = True
    try:
        pre = f"{out}/_pre.onnx"
        try:
            from onnxruntime.quantization.shape_inference import quant_pre_process
            quant_pre_process(fp32, pre)
        except Exception:
            shutil.copy(fp32, pre)
        quantize_static(pre, int8, Calib(), quant_format=QuantFormat.QDQ, per_channel=True,
                        activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8)
        os.remove(pre)
    except Exception as e:
        int8_ok = False; print("La cuantización falló:", e)

    sizes, onnx_m = {}, {}
    ref_name = "val"
    ref_set = val_ds
    holds = [(p, l) for parts in ood.values() for (p, l) in parts["hold"]]
    if holds: ref_name, ref_set = "ood_holdout", FileSet(holds, eval_tf)
    for tag, path in [("fp32", fp32), ("int8", int8)]:
        if tag == "int8" and not int8_ok: continue
        sizes[tag] = round(os.path.getsize(path) / 1e6, 2)
        lg, yy, ms = onnx_logits(path, ref_set)
        onnx_m[tag] = report(f"ONNX {tag} {ref_name} ({sizes[tag]} MB, {ms:.0f} ms/foto en CPU, sin optimizar)", lg, yy, temperature, conf)
        onnx_m[tag]["ms_per_image_colab_cpu"] = round(ms, 1)
        metrics[f"onnx_{tag}_{ref_name}"] = onnx_m[tag]

    drop = onnx_m["fp32"]["accuracy_all"] - onnx_m["int8"]["accuracy_all"] if int8_ok else None
    if int8_ok and drop <= MAX_INT8_DROP and sizes["int8"] <= MAX_MODEL_MB: rec = "int8"
    elif sizes["fp32"] <= MAX_MODEL_MB: rec = "fp32"
    elif int8_ok and sizes["int8"] <= MAX_MODEL_MB:
        rec = "int8"; print(f"AVISO: int8 pierde {drop:.3f} de precisión, pero fp32 pesa {sizes['fp32']} MB (> {MAX_MODEL_MB}).")
    else: rec = None
    print(f"Caída por cuantización: {drop if drop is None else round(drop, 3)} → recomendado: {rec}")
    os.remove(f"{out}/best.pt")
    return {"name": name, "out": out, "mean": mean, "std": std, "temperature": temperature, "conf": conf,
            "metrics": metrics, "sizes": sizes, "recommended": rec, "int8_ok": int8_ok,
            "score": onnx_m[rec]["accuracy_all"] if rec else -1, "score_source": ref_name}

In [ ]:
results = []
for name in MODELS:
    try:
        results.append(run_model(name))
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"\n*** {name} falló: {e}. Se sigue con el siguiente modelo. ***")
    if device == "cuda": torch.cuda.empty_cache()

print("\nmodelo | archivo | MB | precisión ({}) | umbral".format(results[0]["score_source"] if results else "—"))
for r in results:
    mb = r["sizes"].get(r["recommended"], "—") if r["recommended"] else "—"
    print(f"{r['name']} | {r['recommended']} | {mb} | {r['score']:.3f} | {r['conf']}")

## 5. Elegir el mejor modelo, escribir la ficha y descargar

Gana el de mayor precisión fuera de distribución (o de validación si no hubo datos externos) entre los que pesan ≤ 10 MB;
si empatan (±0,5 %), el más liviano.

In [ ]:
def clean(x):
    """NaN/Infinity -> None para que model_card.json sea JSON válido."""
    if isinstance(x, float) and not math.isfinite(x): return None
    if isinstance(x, dict): return {k: clean(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [clean(v) for v in x]
    return x

valid = [r for r in results if r["recommended"]]
assert valid, "Ningún modelo quedó por debajo de 10 MB o todos fallaron. Revisa los mensajes de arriba."
top = max(r["score"] for r in valid)
best = min([r for r in valid if r["score"] >= top - 0.005], key=lambda r: r["sizes"][r["recommended"]])
print("Elegido:", best["name"], best["recommended"])

final = f"{OUT_DIR}/final"; shutil.rmtree(final, ignore_errors=True); os.makedirs(final)
model_file = f"model_{best['recommended']}.onnx"
shutil.copy(f"{best['out']}/{model_file}", f"{final}/{model_file}")
for png in glob.glob(f"{best['out']}/confusion_*.png"): shutil.copy(png, final)

card = {
    "id": f"arabica-v1-{best['name']}",
    "architecture": best["name"],
    "classes": CLASSES,
    "input": {"name": "input", "shape": [1, 3, IMG_SIZE, IMG_SIZE], "layout": "NCHW", "color": "RGB",
              "resize": "estirar a 224x224 (sin recorte)", "scale": "dividir entre 255", "mean": best["mean"], "std": best["std"]},
    "output": {"name": "logits", "postprocess": "softmax(logits / temperature)"},
    "temperature": best["temperature"],
    "unsure_rule": {"min_confidence": best["conf"], "min_margin": MARGIN_THRESHOLD},
    "files": {best["recommended"]: model_file},
    "recommended_file": model_file,
    "size_mb": {best["recommended"]: best["sizes"][best["recommended"]]},
    "metrics": best["metrics"],
    "candidates": [{"model": r["name"], "file": r["recommended"], "size_mb": r["sizes"], "score": r["score"],
                    "score_source": r["score_source"], "min_confidence": r["conf"]} for r in results],
    "training": {"epochs": EPOCHS, "img_size": IMG_SIZE, "max_per_class_jmuben": MAX_PER_CLASS, "seed": SEED,
                 "threshold_rule": f"el umbral más bajo con precisión ≥ {TARGET_PRECISION} y cobertura ≥ {MIN_COVERAGE} en la mitad 'calib' de Uganda+Perú"},
    "training_data": [
        {"name": "JMuBEN / JMuBEN2", "country": "Kenia", "license": "CC BY 4.0",
         "source": "https://huggingface.co/datasets/Project-AgML/arabica_coffee_leaf_disease_classification",
         "doi": ["10.17632/t2r6rszp5c.1", "10.17632/tgv3zb82nd.1"], "images_used": len(train_idx)},
        {"name": "BRACOL", "country": "Brasil", "license": "CC BY 4.0",
         "source": "https://data.mendeley.com/datasets/yy2k5y8mxg/1", "images_used": len(bracol_known)},
    ],
    "evaluation_data_never_trained_on": [
        {"name": "Uganda (Soroti University)", "source": "https://data.mendeley.com/datasets/k36wnd6knb/1"},
        {"name": "Perú (Saposoa)", "source": "https://data.mendeley.com/datasets/mfpxg4y65r/1"},
    ],
    "known_limits": [
        "Solo hojas de arábica; no cubre frutos ni robusta.",
        "JMuBEN son hojas recortadas y con aumentación: la validación interna está inflada.",
        "El número honesto es 'ood_*_holdout': la mitad de Uganda/Perú que no se usó para elegir el umbral.",
        "No reconoce deficiencias nutricionales ni enfermedades fuera de las 5 clases.",
    ],
}
if SMOKE_TEST: card["smoke_test"] = True
json.dump(clean(card), open(f"{final}/model_card.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False, allow_nan=False)

zip_path = shutil.make_archive("jani_model", "zip", final)
print(json.dumps(clean(card["metrics"]), indent=2, ensure_ascii=False))
print("\nListo:", zip_path, "→ descomprimir dentro de app/public/models/arabica-v1/")
try:
    from google.colab import files; files.download(zip_path)
except Exception:
    print("Descarga manual desde el panel de archivos:", zip_path)